https://colab.research.google.com/drive/1S0w3cPRmTPLbNGyKelUBXu9taIp3wpDa?usp=sharing

# **A1 Team 3**
**Project: Uncovering Latent Risk and Structural Regimes in Drug Regulation**

Members: Abhi Jindal, Daaksha B. Arun, Mahesh Wadhokar, Tanya Chhabra

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Data Loading
Loading EMA medicines dataset and identifying key variables.

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/Medicines_output_european_public_assessment_reports.xlsx"
df = pd.read_excel(file_path, header=8)
df = df.loc[:, ~df.columns.astype(str).str.contains("^Unnamed")]

print("Rows:", len(df))
print("Columns:", df.shape[1])
df.head()

In [ ]:
print(df.columns)


## 2. EDA & Preprocessing Updates (M2 → M4)

**M2 baseline:**
- Text normalization (lowercasing, punctuation removal, stopword filtering)
- TF-IDF vectorization
- Cosine similarity computation
- Observed instability and lack of global clustering by authorization status

**M4 refinement:**
- Dataset and regulatory labels remain unchanged
- Introduced embedding-based representation
- Applied both PCA (linear) and UMAP (non-linear) dimensionality reduction

In [ ]:
TEXT_COL = "Condition / indication"

print("Total rows:", len(df))
print("Missing indications:", df[TEXT_COL].isna().sum())
print("Unique indications:", df[TEXT_COL].nunique())

df["text_length_words"] = df[TEXT_COL].fillna("").str.split().str.len()
print(df["text_length_words"].describe())

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOPWORDS = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 2]
    return " ".join(tokens)

df["clean_text"] = df[TEXT_COL].apply(clean_text)

df[["Condition / indication", "clean_text"]].head()

## 3. Analysis & Experiments

This section evaluates whether therapeutic similarity derived from indication text forms patterns that align with regulatory outcomes.

### 3.1 TF-IDF Baseline (Cosine Similarity)

TF-IDF provides a high-dimensional frequency-based representation of indication text.  
Cosine similarity is used to evaluate lexical similarity between medicines.

This baseline tests whether surface-level word similarity aligns with authorization outcomes.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95,
    ngram_range=(1,2)
)

X_tfidf = tfidf.fit_transform(df["clean_text"])

print("TF-IDF shape:", X_tfidf.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

sample_size = min(500, len(df))
sample_idx = np.random.choice(len(df), sample_size, replace=False)

S_tfidf = cosine_similarity(X_tfidf[sample_idx])

sim_values = S_tfidf[np.triu_indices_from(S_tfidf, k=1)]

print(pd.Series(sim_values).describe())

### 3.2 Embedding-Based Representation

Embedding representations capture contextual semantic relationships beyond token frequency.

This refinement evaluates whether deeper semantic encoding improves structural coherence and regulatory differentiation.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

X_emb = model.encode(
    df[TEXT_COL].fillna("").astype(str).tolist(),
    show_progress_bar=True
)

X_emb = np.array(X_emb)

print("Embedding shape:", X_emb.shape)

### 3.3 PCA (Linear Structure)

Principal Component Analysis (PCA) is applied to scaled embedding features to inspect dominant linear semantic directions.

The first two components explain approximately 9.5% of total variance, indicating that semantic information is distributed across many dimensions.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

X_emb_scaled = StandardScaler().fit_transform(X_emb)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_emb_scaled)

plt.scatter(X_pca[:,0], X_pca[:,1], s=10)
plt.title("Embedding Representation (PCA)")
plt.show()

**Figure 3.1.** PCA projection of embedding representations colored by authorization status. Substantial overlap across regulatory categories indicates no dominant linear axis separating approval outcomes.

### 3.4 UMAP (Non-Linear Structure)

UMAP is applied to assess local neighborhood structure in the embedding space.  
Unlike PCA, UMAP preserves local relationships and may reveal tighter therapeutic pockets.

In [ ]:
import umap
import matplotlib.pyplot as plt

reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_emb)

plt.scatter(X_umap[:,0], X_umap[:,1], s=10)
plt.title("Embedding Representation (UMAP)")
plt.show()

**Figure 3.2.** UMAP projection of embedding-based representations colored by authorization status.
While local therapeutic pockets are visible, regulatory categories remain interspersed within clusters, indicating that non-linear structure does not produce regulatory separation.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd
import matplotlib.pyplot as plt

LABEL_COL = "Authorisation status"

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_emb)

pca_50 = PCA(n_components=50, random_state=42)
X_reduced = pca_50.fit_transform(X_scaled)

print("Reduced shape:", X_reduced.shape)

### 3.5 K-Means Model Selection

K-means clustering is applied to the PCA-reduced embedding space (50 components).

A grid search over k = {3, 5, 8, 12, 16} evaluates structural cohesion using silhouette scores.

In [ ]:
k_values = [3, 5, 8, 12, 16]
results = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_reduced)
    sil = silhouette_score(X_reduced, labels)
    results.append({"k": k, "silhouette_score": sil})

results_df = pd.DataFrame(results)
display(results_df)

**Table 3.1.** Silhouette scores across tested k values.

In [ ]:
best_k = results_df.loc[results_df["silhouette_score"].idxmax(), "k"]
print("Best k selected:", best_k)

kmeans = KMeans(n_clusters=int(best_k), random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_reduced)

### 3.6 Cluster Composition by Authorization Status (k = 16)

Cluster composition is examined to evaluate whether semantic clusters correspond to regulatory outcomes.

Authorized medicines remain the majority across all clusters, and withdrawn/refused medicines are distributed across clusters rather than forming isolated groups.

In [ ]:
print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index())

cluster_summary = pd.crosstab(df["cluster"], df[LABEL_COL], normalize="index")
display(cluster_summary)

**Table 3.2.** Cluster-level distribution of authorization status.

In [ ]:
cluster_comp.plot(
    kind='bar',
    stacked=True,
    figsize=(12,6)
)

plt.title("Cluster Composition by Regulatory Status")
plt.xlabel("Cluster")
plt.ylabel("Proportion")
plt.legend(title="Authorisation Status")
plt.tight_layout()
plt.show()

**Figure 3.2.** Cluster composition by authorization status (k = 16). All clusters contain mixed regulatory outcomes.

In [ ]:
pca_2 = PCA(n_components=2, random_state=42)
X_2d = pca_2.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_2d[:,0], X_2d[:,1], c=df["cluster"], cmap="tab10", s=10)
plt.title("K-Means Clusters (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
label_codes = pd.Categorical(df[LABEL_COL]).codes

plt.figure(figsize=(8,6))
plt.scatter(X_2d[:,0], X_2d[:,1], c=label_codes, s=10)
plt.title("PCA Projection Colored by Authorisation status")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
stability_scores = []

for seed in [1, 7, 21, 42, 99]:
    km = KMeans(n_clusters=int(best_k), random_state=seed, n_init=10)
    labels = km.fit_predict(X_reduced)
    sil = silhouette_score(X_reduced, labels)
    stability_scores.append(sil)

print("Silhouette scores across seeds:", stability_scores)

### 3.7 Stability Across Random Seeds

Clustering with k = 16 is repeated across multiple random seeds.

Silhouette scores range approximately from 0.355 to 0.365 (mean ≈ 0.361, SD ≈ 0.0035), indicating stable structural partitions despite regulatory mixing.

## 4. Findings & Interpretations

Semantic structure is clearly present in EMA indication text, with medicines organizing into coherent therapeutic groupings. However, these groupings do not align with regulatory outcomes.

Approved, withdrawn, and refused medicines remain interspersed across semantic clusters, suggesting that therapeutic similarity alone does not explain authorization decisions.

Regulatory outcomes likely depend on broader clinical, evidentiary, and organizational factors beyond textual similarity.

## 5. Final Deliverables Summary

This notebook evaluates whether therapeutic similarity in EMA indication text differentiates regulatory outcomes.

Key components included:
- TF-IDF baseline representation and cosine similarity
- Embedding-based semantic representation (M4 refinement)
- Linear (PCA) and non-linear (UMAP) structural inspection
- K-means clustering with grid search over k values
- Cluster composition analysis by authorization status
- Stability analysis across random seeds

Summary conclusion:
While therapeutic structure is clearly present in indication text, semantic clustering does not produce regulatory separation. Approved, withdrawn, and refused medicines remain interspersed across therapeutic clusters. This suggests that therapeutic similarity alone is insufficient to explain EMA authorization outcomes.